# Scaled Dot-Product Attention


In [1]:
import math
from pathlib import Path

import torch
import torch.nn.functional as F

torch.manual_seed(0)

LECTURE_NOTE_PATH = Path(
    "/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Scaled Dot-Product Attention.md"
)

def manual_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
    scale: float | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if scale is None:
        scale = math.sqrt(q.size(-1))
    scores = q @ k.T / scale
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    probs = torch.softmax(scores, dim=-1)
    out = probs @ v
    return scores, probs, out

print(LECTURE_NOTE_PATH)


/Users/montekkundan/Library/Mobile Documents/iCloud~md~obsidian/Documents/lectures/LLM/concepts/Scaled Dot-Product Attention.md


## Demo A: compute QK^T, softmax, and weighted sum


In [2]:
q = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
k = torch.tensor([[1.0, 0.0], [0.8, 0.2], [0.0, 1.0]])
v = torch.tensor([[10.0, 0.0], [7.0, 1.0], [0.0, 10.0]])

scores, probs, out = manual_attention(q, k, v)
print('scores:')
print(scores)
print('attention probabilities:')
print(probs)
print('output:')
print(out)


scores:
tensor([[0.7071, 0.5657, 0.0000],
        [0.0000, 0.1414, 0.7071]])
attention probabilities:
tensor([[0.4235, 0.3677, 0.2088],
        [0.2392, 0.2756, 0.4852]])
output:
tensor([[6.8088, 2.4559],
        [4.3214, 5.1275]])


## Demo B: masking blocks illegal positions


In [3]:
x = torch.tensor(
    [
        [1.0, 0.0, 0.0, 1.0],
        [1.0, 1.0, 0.0, 0.0],
        [0.0, 1.0, 1.0, 0.0],
        [0.0, 0.0, 1.0, 1.0],
    ]
)
causal_mask = torch.tril(torch.ones(x.size(0), x.size(0), dtype=torch.bool))
_, probs, _ = manual_attention(x, x, x, mask=causal_mask)

print('causal mask:')
print(causal_mask.int())
print('causal attention probabilities:')
print(probs)


causal mask:
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
causal attention probabilities:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.3775, 0.6225, 0.0000, 0.0000],
        [0.1863, 0.3072, 0.5065, 0.0000],
        [0.2350, 0.1425, 0.2350, 0.3875]])


## Demo C: why divide by sqrt(d_k)


In [4]:
for d_k in [4, 16, 64, 256]:
    q = torch.randn(1024, d_k)
    k = torch.randn(1024, d_k)
    raw_logits = (q * k).sum(dim=-1)
    scaled_logits = raw_logits / math.sqrt(d_k)
    print(
        f'd_k={d_k:>3} | raw std={raw_logits.std():.3f} | scaled std={scaled_logits.std():.3f}'
    )


d_k=  4 | raw std=1.929 | scaled std=0.964
d_k= 16 | raw std=4.061 | scaled std=1.015
d_k= 64 | raw std=7.804 | scaled std=0.976
d_k=256 | raw std=16.316 | scaled std=1.020


## Demo D: softmax temperature and saturation


In [5]:
logits = torch.tensor([0.2, 0.4, 1.5, 3.0])
for temperature in [0.5, 1.0, 2.0]:
    probs = torch.softmax(logits / temperature, dim=-1)
    entropy = -(probs * torch.log(probs)).sum()
    print(f'temperature={temperature:.1f} -> probs={probs.numpy()} entropy={entropy:.3f}')


temperature=0.5 -> probs=[0.00349184 0.00520921 0.04701322 0.94428575] entropy=0.245
temperature=1.0 -> probs=[0.04477209 0.05468475 0.16428207 0.7362611 ] entropy=0.820
temperature=2.0 -> probs=[0.12382504 0.13684784 0.2371919  0.5021353 ] entropy=1.218


## Demo E: exact PyTorch SDPA matches the manual formula


In [6]:
q = torch.randn(1, 1, 4, 8)
k = torch.randn(1, 1, 4, 8)
v = torch.randn(1, 1, 4, 8)
mask = torch.tril(torch.ones(4, 4, dtype=torch.bool))

_, _, manual = manual_attention(q[0, 0], k[0, 0], v[0, 0], mask=mask)
builtin = F.scaled_dot_product_attention(q, k, v, attn_mask=mask).squeeze(0).squeeze(0)

print('manual == torch.nn.functional.scaled_dot_product_attention:', torch.allclose(manual, builtin, atol=1e-6))


manual == torch.nn.functional.scaled_dot_product_attention: True


## Rasbt and nanochat


In [7]:
references = {
    'rasbt': {
        'files': [
            'https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb',
        ],
        'core': 'Rasbt writes out the score matrix, softmax, masking, and weighted sum step by step so students can inspect every tensor.',
    },
    'nanochat': {
        'files': [
            'https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py',
            'https://github.com/karpathy/nanochat/blob/master/nanochat/flash_attention.py',
        ],
        'core': 'nanochat keeps the same math but routes it through optimized kernels and FlashAttention-style implementation paths.',
    },
}

for name, info in references.items():
    print(name.upper())
    print('core :', info['core'])
    print('files:')
    for file in info['files']:
        print(' -', file)
    print()


RASBT
core : Rasbt writes out the score matrix, softmax, masking, and weighted sum step by step so students can inspect every tensor.
files:
 - https://github.com/rasbt/LLMs-from-scratch/blob/main/ch03/01_main-chapter-code/ch03.ipynb

NANOCHAT
core : nanochat keeps the same math but routes it through optimized kernels and FlashAttention-style implementation paths.
files:
 - https://github.com/karpathy/nanochat/blob/master/nanochat/gpt.py
 - https://github.com/karpathy/nanochat/blob/master/nanochat/flash_attention.py

